# Churn Baseline Notebook
RabTech Academy - ML Problem Framing & Responsible Data Card

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Update the filename if needed
path = 'customer-churn-training.csv'
df = pd.read_csv(path)
df.head()


In [ ]:
print(df.shape)
print(df.isna().sum())
print('Duplicate rows:', df.duplicated().sum())
print('Duplicate IDs:', df['customer_id'].duplicated().sum())
print(df['churned'].value_counts())


In [ ]:
features = ['tenure_months','support_tickets','monthly_spend_inr','last_login_days','plan_type']
X = df[features]
y = df['churned']

preprocess = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), ['plan_type'])
], remainder='passthrough')

model = Pipeline([
    ('prep', preprocess),
    ('clf', LogisticRegression(max_iter=1000))
])

# This split is only a demonstration because n=12 is far too small for reliable evaluation.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
model.fit(X_train, y_train)
pred = model.predict(X_test)
prob = model.predict_proba(X_test)[:,1]

print('Accuracy:', accuracy_score(y_test, pred))
print('Precision:', precision_score(y_test, pred, zero_division=0))
print('Recall:', recall_score(y_test, pred, zero_division=0))
print('F1:', f1_score(y_test, pred, zero_division=0))
print('ROC-AUC:', roc_auc_score(y_test, prob) if y_test.nunique()==2 else 'N/A: test set has one class')


## Interpretation
This model is a baseline demonstration only. The source dataset contains 12 records, so performance estimates from a train/test split are unstable. Production decisions require a substantially larger dataset, a time-aware validation strategy, leakage checks, calibration, fairness analysis, and review of false positives and false negatives.